In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import kurtosis, skew
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import warnings
warnings.filterwarnings("ignore")
print("Libraries loaded!")

In [ ]:
df = pd.read_csv("train_features.csv")
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Bearings: {df['bearing'].unique()}")
print(df.head(3))

In [ ]:
df["RUL_norm"]      = df["RUL"] / df["total_steps"]
df["life_fraction"] = df["time_step"] / df["total_steps"]

print("RUL normalized to [0, 1]")
print(df[["bearing", "RUL", "total_steps", "RUL_norm"]].head())

In [ ]:
feature_cols = [
    "rms_x", "peak_x", "kurtosis_x", "skew_x", "std_x", "crest_x",
    "rms_y", "peak_y", "kurtosis_y", "skew_y", "std_y", "crest_y"
]

X      = df[feature_cols].values
y      = df["RUL_norm"].values
groups = df["bearing"].values

print(f"Features: {len(feature_cols)}")
print(f"Samples:  {len(X)}")
print(f"Feature list: {feature_cols}")

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

print(f"Train bearings : {set(groups[train_idx])}")
print(f"Val bearings   : {set(groups[val_idx])}")
print(f"Overlap        : {set(groups[train_idx]) & set(groups[val_idx])} ← must be empty")
print(f"Train samples  : {len(X_train)}")
print(f"Val samples    : {len(X_val)}")

In [ ]:
model_results = {}

# 1. Random Forest
print("Training Random Forest...")
rf = RandomForestRegressor(
    n_estimators=300, max_depth=8,
    min_samples_leaf=10, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_r2   = r2_score(y_val, rf.predict(X_val))
rf_rmse = np.sqrt(mean_squared_error(y_val, rf.predict(X_val)))
rf_mae  = mean_absolute_error(y_val, rf.predict(X_val))
model_results["Random Forest"] = {"model": rf, "r2": rf_r2, "rmse": rf_rmse, "mae": rf_mae}
print(f"  R²: {rf_r2:.4f} | RMSE: {rf_rmse:.4f} | MAE: {rf_mae:.4f}")

# 2. Extra Trees
print("Training Extra Trees...")
et = ExtraTreesRegressor(
    n_estimators=300, max_depth=8,
    min_samples_leaf=10, random_state=42, n_jobs=-1
)
et.fit(X_train, y_train)
et_r2   = r2_score(y_val, et.predict(X_val))
et_rmse = np.sqrt(mean_squared_error(y_val, et.predict(X_val)))
et_mae  = mean_absolute_error(y_val, et.predict(X_val))
model_results["Extra Trees"] = {"model": et, "r2": et_r2, "rmse": et_rmse, "mae": et_mae}
print(f"  R²: {et_r2:.4f} | RMSE: {et_rmse:.4f} | MAE: {et_mae:.4f}")

# 3. Gradient Boosting
print("Training Gradient Boosting...")
gb = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05,
    max_depth=4, random_state=42
)
gb.fit(X_train, y_train)
gb_r2   = r2_score(y_val, gb.predict(X_val))
gb_rmse = np.sqrt(mean_squared_error(y_val, gb.predict(X_val)))
gb_mae  = mean_absolute_error(y_val, gb.predict(X_val))
model_results["Gradient Boosting"] = {"model": gb, "r2": gb_r2, "rmse": gb_rmse, "mae": gb_mae}
print(f"  R²: {gb_r2:.4f} | RMSE: {gb_rmse:.4f} | MAE: {gb_mae:.4f}")

# 4. SVR
print("Training SVR...")
svr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf", C=10, epsilon=0.1))
])
svr_pipe.fit(X_train, y_train)
svr_r2   = r2_score(y_val, svr_pipe.predict(X_val))
svr_rmse = np.sqrt(mean_squared_error(y_val, svr_pipe.predict(X_val)))
svr_mae  = mean_absolute_error(y_val, svr_pipe.predict(X_val))
model_results["SVR"] = {"model": svr_pipe, "r2": svr_r2, "rmse": svr_rmse, "mae": svr_mae}
print(f"  R²: {svr_r2:.4f} | RMSE: {svr_rmse:.4f} | MAE: {svr_mae:.4f}")

# Summary table
print("\n" + "="*55)
print(f"{'Model':<20} {'R²':>8} {'RMSE':>8} {'MAE':>8}")
print("="*55)
for name, res in model_results.items():
    print(f"{name:<20} {res['r2']:>8.4f} {res['rmse']:>8.4f} {res['mae']:>8.4f}")
print("="*55)

In [ ]:
best_name  = max(model_results, key=lambda k: model_results[k]["r2"])
best_model = model_results[best_name]["model"]
best_r2    = model_results[best_name]["r2"]

print(f"✅ Best model: {best_name} (R²={best_r2:.4f})")

# Retrain on ALL training data
best_model.fit(X, y)
joblib.dump(best_model, "best_rul_model_v2.pkl")
joblib.dump(feature_cols, "feature_cols_v2.pkl")
print("Saved: best_rul_model_v2.pkl")

In [ ]:
def extract_features(filepath):
    df_raw = pd.read_csv(filepath, header=None)
    ax = df_raw[4].values
    ay = df_raw[5].values if 5 in df_raw.columns else ax
    features = {}
    for name, sig in [("x", ax), ("y", ay)]:
        features[f"rms_{name}"]      = np.sqrt(np.mean(sig**2))
        features[f"peak_{name}"]     = np.max(np.abs(sig))
        features[f"kurtosis_{name}"] = kurtosis(sig)
        features[f"skew_{name}"]     = skew(sig)
        features[f"std_{name}"]      = np.std(sig)
        features[f"crest_{name}"]    = np.max(np.abs(sig)) / (np.sqrt(np.mean(sig**2)) + 1e-10)
    return features

test_path = "Test_set"
all_test  = []
for bearing in sorted(os.listdir(test_path)):
    path  = os.path.join(test_path, bearing)
    files = sorted([f for f in os.listdir(path) if f.startswith("acc_")])
    records = []
    for i, f in enumerate(files):
        feats = extract_features(os.path.join(path, f))
        feats["time_step"]   = i
        feats["total_steps"] = len(files)
        feats["bearing"]     = bearing
        records.append(feats)
    all_test.append(pd.DataFrame(records))
    print(f"  {bearing}: {len(files)} files")

test_df = pd.concat(all_test, ignore_index=True)
print(f"\nTotal test rows: {test_df.shape[0]}")

In [ ]:
X_test = test_df[feature_cols].values

# Per-bearing uncertainty
if best_name in ["Random Forest", "Extra Trees"]:
    preds_trees = np.array([tree.predict(X_test) for tree in best_model.estimators_])
    y_pred_norm = preds_trees.mean(axis=0)
    y_std_norm  = preds_trees.std(axis=0)
else:
    y_pred_norm = best_model.predict(X_test)
    y_std_norm  = np.zeros(len(y_pred_norm))
    for bearing in test_df["bearing"].unique():
        mask   = test_df["bearing"] == bearing
        preds  = y_pred_norm[mask]
        y_std_norm[mask] = pd.Series(preds).rolling(10, min_periods=1).std().fillna(0).values

# Smooth predictions with rolling average
smoothed_pred = []
smoothed_std  = []
for bearing in test_df["bearing"].unique():
    mask  = test_df["bearing"] == bearing
    preds = pd.Series(y_pred_norm[mask]).rolling(20, min_periods=1).mean().values
    stds  = pd.Series(y_std_norm[mask]).rolling(20, min_periods=1).mean().values
    smoothed_pred.extend(preds)
    smoothed_std.extend(stds)

test_df["pred_norm"] = np.clip(smoothed_pred, 0, 1)
test_df["std_norm"]  = smoothed_std

# Convert to seconds
bearing_total = test_df.groupby("bearing")["total_steps"].first()
test_df["Predicted_RUL_s"] = test_df["pred_norm"] * test_df["bearing"].map(bearing_total) * 10
test_df["Uncertainty_s"]   = test_df["std_norm"]  * test_df["bearing"].map(bearing_total) * 10
test_df["Lower_RUL_s"]     = np.maximum(test_df["Predicted_RUL_s"] - test_df["Uncertainty_s"], 0)
test_df["Upper_RUL_s"]     = test_df["Predicted_RUL_s"] + test_df["Uncertainty_s"]

def classify_health(rul_norm):
    if rul_norm <= 0.20:
        return "Imminent failure"
    elif rul_norm <= 0.50:
        return "Wear detectable"
    else:
        return "Non-critical"

test_df["Health_State"] = test_df["pred_norm"].apply(classify_health)
print("Predictions done!")

In [ ]:
actual_rul = {
    "Bearing1_3": 5730, "Bearing1_4": 339,  "Bearing1_5": 1610,
    "Bearing1_6": 1460, "Bearing1_7": 7570, "Bearing2_3": 7530,
    "Bearing2_4": 1390, "Bearing2_5": 3090, "Bearing2_6": 1290,
    "Bearing2_7": 580,  "Bearing3_3": 820
}

latest = test_df.sort_values("time_step").groupby("bearing").last().reset_index()

print(f"Best model: {best_name}\n")
print(f"{'Bearing':<12} {'Predicted(s)':>13} {'Actual(s)':>10} {'Error%':>8} {'Uncertainty':>13} {'Health'}")
print("-" * 85)

errors = []
for _, row in latest.iterrows():
    b      = row["bearing"]
    pred_s = row["Predicted_RUL_s"]
    act_s  = actual_rul[b]
    unc_s  = row["Uncertainty_s"]
    health = row["Health_State"]
    err    = abs(pred_s - act_s) / act_s * 100
    errors.append(err)
    print(f"{b:<12} {pred_s:>13.0f} {act_s:>10} {err:>7.1f}% {unc_s:>12.0f}s  {health}")

print("-" * 85)
print(f"Average Error: {np.mean(errors):.1f}%")
print(f"Best  Error:   {np.min(errors):.1f}%")
print(f"Worst Error:   {np.max(errors):.1f}%")

In [ ]:
test_df.to_csv("arya_test_predictions_timeseries_v2.csv", index=False)
latest.to_csv("arya_test_predictions_final_v2.csv", index=False)
print("Saved: arya_test_predictions_timeseries_v2.csv")
print("Saved: arya_test_predictions_final_v2.csv")
print(f"\n✅ Modelling v2 complete! Best model: {best_name}")